In [2]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.preprocessing import LabelEncoder

from sklearn.pipeline import Pipeline

In [3]:
columns = [
    'duration',
    'protocol_type',
    'service',
    'flag',
    'src_bytes',
    'dst_bytes',
    'land',
    'wrong_fragment',
    'urgent',
    'hot',
    'num_failed_logins',
    'logged_in',
    'num_compromised',
    'root_shell',
    'su_attempted',
    'num_root',
    'num_file_creations',
    'num_shells',
    'num_access_files',
    'num_outbound_cmds',
    'is_host_login',
    'is_guest_login',
    'count',
    'srv_count',
    'serror_rate',
    'srv_serror_rate',
    'rerror_rate',
    'srv_rerror_rate',
    'same_srv_rate',
    'diff_srv_rate',
    'srv_diff_host_rate',
    'dst_host_count',
    'dst_host_srv_count',
    'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate',
    'dst_host_srv_serror_rate',
    'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate',
    'label',
    'difficulty'
]

train = pd.read_csv(
    "../data/KDDTrain+.txt",
    names=columns
)

print(train.shape)
print(train.head())

(125973, 43)
   duration protocol_type   service flag  src_bytes  dst_bytes  land  \
0         0           tcp  ftp_data   SF        491          0     0   
1         0           udp     other   SF        146          0     0   
2         0           tcp   private   S0          0          0     0   
3         0           tcp      http   SF        232       8153     0   
4         0           tcp      http   SF        199        420     0   

   wrong_fragment  urgent  hot  ...  dst_host_same_srv_rate  \
0               0       0    0  ...                    0.17   
1               0       0    0  ...                    0.00   
2               0       0    0  ...                    0.10   
3               0       0    0  ...                    1.00   
4               0       0    0  ...                    1.00   

   dst_host_diff_srv_rate  dst_host_same_src_port_rate  \
0                    0.03                         0.17   
1                    0.60                         0.88   
2

In [4]:
train_df = pd.read_csv(
    "../data/KDDTrain+.txt",
    names=columns
)

test_df = pd.read_csv(
    "../data/KDDTest+.txt",
    names=columns
)

In [5]:
train_df.drop(columns=["difficulty"], inplace=True)

test_df.drop(columns=["difficulty"], inplace=True)

In [6]:
X_train = train_df.drop(columns=["label"])

y_train = train_df["label"]


X_test = test_df.drop(columns=["label"])

y_test = test_df["label"]

In [7]:
categorical_features = [
    "protocol_type",
    "service",
    "flag"
]

In [8]:
numeric_features = X_train.columns.difference(
    categorical_features
).tolist()

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "num",
            RobustScaler(),
            numeric_features
        )
    ]
)

In [10]:
# Apply preprocessing

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [11]:
# Get feature names

feature_names = preprocessor.get_feature_names_out()

# Convert to DataFrame

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

In [12]:
# Convert labels to Binary Classification

y_train_binary = y_train.copy()
y_test_binary = y_test.copy()

y_train_binary = y_train_binary.apply(
    lambda x: 0 if x == "normal" else 1
)

y_test_binary = y_test_binary.apply(
    lambda x: 0 if x == "normal" else 1
)

In [13]:
print("Train")
print(y_train_binary.value_counts())

print()

print("Test")
print(y_test_binary.value_counts())

Train
label
0    67343
1    58630
Name: count, dtype: int64

Test
label
1    12833
0     9711
Name: count, dtype: int64


In [14]:
print(X_train_processed.shape)
print(X_test_processed.shape)

print(y_train_binary.shape)
print(y_test_binary.shape)

(125973, 122)
(22544, 122)
(125973,)
(22544,)


In [15]:
X_train_processed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125973 entries, 0 to 125972
Columns: 122 entries, cat__protocol_type_icmp to num__wrong_fragment
dtypes: float64(122)
memory usage: 117.3 MB


In [16]:
import joblib

joblib.dump(X_train_processed, "../data/X_train.pkl")
joblib.dump(X_test_processed, "../data/X_test.pkl")

joblib.dump(y_train_binary, "../data/y_train.pkl")
joblib.dump(y_test_binary, "../data/y_test.pkl")

['../data/y_test.pkl']